In [1]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

import os
import sys
import time 

from pathlib import Path
from loguru import logger

import duckdb

In [2]:
ROOT = Path.cwd().parent
RAW_DATA_DIR = ROOT / "data" / "raw"
OUTPUT_DATA_DIR = ROOT / "data" / "interim"
OUTPUT_DATA_DIR.mkdir(parents=True, exist_ok=True)

In [3]:
con = duckdb.connect()
csv_files = sorted(RAW_DATA_DIR.glob("*.csv"))
for csv_file in csv_files:
    parquet_file = OUTPUT_DATA_DIR / f"{csv_file.stem}.parquet"

    # Skip conversion if the Parquet file already exists
    if parquet_file.exists():
        logger.info(f"Skipping {csv_file} as {parquet_file} already exists.")
        continue

    start_time = pd.Timestamp.now()
    con.execute(f"COPY (SELECT * FROM read_csv_auto('{csv_file}')) TO '{parquet_file}' (FORMAT PARQUET);")
    end_time = pd.Timestamp.now()

    logger.info(f"Converted {csv_file} to {parquet_file} in {end_time - start_time}")
    old_size = csv_file.stat().st_size/ (1024 * 1024)
    new_size = parquet_file.stat().st_size/ (1024 * 1024)
    logger.info(f"Size reduced from {old_size:.2f} MB to {new_size:.2f} MB ({(old_size - new_size) / old_size * 100:.2f}% reduction)")

con.close()


2026-10-11 10:51:25.560 | INFO     | __main__:<module>:8 - Skipping /Users/yuzhezhang/Library/Mobile Documents/com~apple~CloudDocs/MyProject/data_science/data/raw/clients.csv as /Users/yuzhezhang/Library/Mobile Documents/com~apple~CloudDocs/MyProject/data_science/data/interim/clients.parquet already exists.
2026-10-11 10:51:25.560 | INFO     | __main__:<module>:8 - Skipping /Users/yuzhezhang/Library/Mobile Documents/com~apple~CloudDocs/MyProject/data_science/data/raw/products.csv as /Users/yuzhezhang/Library/Mobile Documents/com~apple~CloudDocs/MyProject/data_science/data/interim/products.parquet already exists.
2026-10-11 10:51:25.561 | INFO     | __main__:<module>:8 - Skipping /Users/yuzhezhang/Library/Mobile Documents/com~apple~CloudDocs/MyProject/data_science/data/raw/purchases.csv as /Users/yuzhezhang/Library/Mobile Documents/com~apple~CloudDocs/MyProject/data_science/data/interim/purchases.parquet already exists.
2026-10-11 10:51:25.561 | INFO     | __main__:<module>:8 - Skipping

# Checking the Quality of the Data

After the conversion is finished, the next step is to check the quality of the data. In the real world, data is often dirty: some values are missing, some are incorrect, and some columns have the wrong data types.

In [4]:
con = duckdb.connect()

parquet_files = sorted(OUTPUT_DATA_DIR.glob("*.parquet"))

summary = []
for parquet_file in parquet_files:
    relation = con.read_parquet(str(parquet_file))

    summary.append({
        "table": parquet_file.stem,
        "rows": relation.shape[0],
        "columns": relation.shape[1],
    })

summary_df = pd.DataFrame(summary)
summary_df

,table,rows,columns
0,clients,400162,5
1,products,43038,11
2,purchases,45786568,13
3,uplift_sample_submission,200123,2
4,uplift_test,200123,1
5,uplift_train,200039,3


In [5]:
for parquet_file in parquet_files:
    relation = con.read_parquet(str(parquet_file))

    print("=" * 30)
    print(f"Table: {parquet_file.stem}")
    schema_df = pd.DataFrame({
        "column_name": relation.columns,
        "data_type": [str(dtype) for dtype in relation.dtypes]
    })

    display(schema_df)

Table: clients


,column_name,data_type
0,client_id,VARCHAR
1,first_issue_date,TIMESTAMP
2,first_redeem_date,TIMESTAMP
3,age,BIGINT
4,gender,VARCHAR


Table: products


,column_name,data_type
0,product_id,VARCHAR
1,level_1,VARCHAR
2,level_2,VARCHAR
3,level_3,VARCHAR
4,level_4,VARCHAR
5,segment_id,DOUBLE
6,brand_id,VARCHAR
7,vendor_id,VARCHAR
8,netto,DOUBLE
9,is_own_trademark,BIGINT


Table: purchases


...

,column_name,data_type
0,client_id,VARCHAR
1,uplift,DOUBLE


Table: uplift_test


,column_name,data_type
0,client_id,VARCHAR


Table: uplift_train


,column_name,data_type
0,client_id,VARCHAR
1,treatment_flg,BIGINT
2,target,BIGINT


In [12]:
# Check the statistics of the tables
def profile_dataframe(df):
    profile = pd.DataFrame(index=df.columns)

    profile["data_type"] = df.dtypes.astype(str)
    profile["num_missing"] = df.isna().sum()
    profile["pct_missing"] = df.isna().mean() * 100
    profile["num_unique"] = df.nunique(dropna=True)

    profile["min"] = df.min(numeric_only=True)
    profile["max"] = df.max(numeric_only=True)
    profile["mean"] = df.mean(numeric_only=True)
    profile["std"] = df.std(numeric_only=True)

    return profile.rename_axis("column_name").reset_index()

In [15]:
frame_list = ['clients.parquet', 'products.parquet',
              'purchases.parquet', 'uplift_train.parquet']

for parquet_file in frame_list:
    parquet_path = OUTPUT_DATA_DIR / parquet_file
    df = pd.read_parquet(parquet_path)
    profile_df = profile_dataframe(df)

    print("=" * 30)
    print(f"Table: {parquet_file}")
    display(profile_df)

Table: clients.parquet


,column_name,data_type,num_missing,pct_missing,num_unique,min,max,mean,std
0,client_id,str,0,0.00000,400162,NaN,NaN,NaN,NaN
1,first_issue_date,datetime64[us],0,0.00000,396725,NaN,NaN,NaN,NaN
2,first_redeem_date,datetime64[us],35469,8.86366,362527,NaN,NaN,NaN,NaN
3,age,int64,0,0.00000,290,-7491.0,1901.0,46.488112,43.871218
4,gender,str,0,0.00000,3,NaN,NaN,NaN,NaN


Table: products.parquet


,column_name,data_type,num_missing,pct_missing,num_unique,min,max,mean,std
0,product_id,str,0,0.000000,43038,NaN,NaN,NaN,NaN
1,level_1,str,3,0.006971,3,NaN,NaN,NaN,NaN
2,level_2,str,3,0.006971,42,NaN,NaN,NaN,NaN
3,level_3,str,3,0.006971,201,NaN,NaN,NaN,NaN
4,level_4,str,3,0.006971,790,NaN,NaN,NaN,NaN
5,segment_id,float64,1572,3.652586,116,1.0,321.0,153.918222,82.271376
6,brand_id,str,5200,12.082346,4296,NaN,NaN,NaN,NaN
7,vendor_id,str,34,0.079000,3193,NaN,NaN,NaN,NaN
8,netto,float64,3,0.006971,780,0.0,1150.0,0.536966,8.274367
9,is_own_trademark,int64,0,0.000000,2,0.0,1.0,0.035178,0.184232


Table: purchases.parquet


,column_name,data_type,num_missing,pct_missing,num_unique,min,max,mean,std
0,client_id,str,0,0.000000,400162,NaN,NaN,NaN,NaN
1,transaction_id,str,0,0.000000,8045201,NaN,NaN,NaN,NaN
2,transaction_datetime,datetime64[us],0,0.000000,4260251,NaN,NaN,NaN,NaN
3,regular_points_received,float64,0,0.000000,2224,0.0,2399.00,8.049876,12.685028
4,express_points_received,float64,0,0.000000,13,0.0,300.00,0.060841,2.426226
5,regular_points_spent,float64,0,0.000000,1147,-5066.0,0.00,-5.312626,36.036463
6,express_points_spent,float64,0,0.000000,78,-300.0,0.00,-0.318105,3.287976
7,purchase_sum,float64,0,0.000000,258231,0.0,35149.04,777.521477,796.535010
8,store_id,str,0,0.000000,13882,NaN,NaN,NaN,NaN
9,product_id,str,0,0.000000,42530,NaN,NaN,NaN,NaN


Table: uplift_train.parquet


,column_name,data_type,num_missing,pct_missing,num_unique,min,max,mean,std
0,client_id,str,0,0.0,200039,NaN,NaN,NaN,NaN
1,treatment_flg,int64,0,0.0,2,0.0,1.0,0.499808,0.500001
2,target,int64,0,0.0,2,0.0,1.0,0.619889,0.485415


In [16]:
clients = pd.read_parquet(OUTPUT_DATA_DIR / "clients.parquet")

display(clients.head(10))

print("Rows:", len(clients))
print("Unique IDs:", clients["client_id"].nunique())
print("Null IDs:", clients["client_id"].isna().sum())
print("Duplicate IDs:", clients["client_id"].duplicated().sum())

,client_id,first_issue_date,first_redeem_date,age,gender
0,000012768d,2017-08-05 15:40:48,2018-01-04 19:30:07,45,U
1,000036f903,2017-04-10 13:54:23,2017-04-23 12:37:56,72,F
2,000048b7a6,2018-12-15 13:33:11,NaT,68,F
3,000073194a,2017-05-23 12:56:14,2017-11-24 11:18:01,60,F
4,00007c7133,2017-05-22 16:17:08,2018-12-31 17:17:33,67,U
5,00007f9014,2017-08-22 13:35:01,2019-02-15 19:19:18,45,F
6,0000a90cf7,2017-05-20 12:34:50,2017-07-08 12:48:22,45,U
7,0000b59cec,2017-06-09 10:04:20,2017-12-27 18:35:58,46,U
8,0000bb4e4e,2017-05-09 16:02:07,2017-08-25 17:03:01,36,M
9,0000bcec9c,2018-09-29 18:18:20,2019-09-29 11:52:58,34,U


Rows: 400162
Unique IDs: 400162
Null IDs: 0
Duplicate IDs: 0


In [18]:
products = pd.read_parquet(OUTPUT_DATA_DIR / "products.parquet")

display(products.head(10))


,product_id,level_1,level_2,level_3,level_4,segment_id,brand_id,vendor_id,netto,is_own_trademark,is_alcohol
0,0003020d3c,c3d3a8e8c6,c2a3ea8d5e,b7cda0ec0c,6376f2a852,123.0,394a54a7c1,9eaff48661,0.400,0,0
1,0003870676,e344ab2e71,52f13dac0c,d3cfe81323,6dc544533f,105.0,acd3dd483f,10486c3cf0,0.680,0,0
2,0003ceaf69,c3d3a8e8c6,f2333c90fb,419bc5b424,f6148afbc0,271.0,f597581079,764e660dda,0.500,0,0
3,000701e093,ec62ce61e3,4202626fcb,88a515c084,48cf3d488f,172.0,54a90fe769,03c2d70bad,0.112,0,0
4,0007149564,e344ab2e71,52f13dac0c,d3cfe81323,6dc544533f,105.0,63417fe1f3,f329130198,0.600,0,0
5,0008bf7786,e344ab2e71,749c619457,5e89a8adcc,d453d4e1e8,165.0,0d6f137fb6,88f2ca925a,0.750,0,1
6,0009170c33,c3d3a8e8c6,ad2b2e17d2,ca69ed9de2,8bbeabc581,212.0,NaN,6a48b9976d,0.400,0,0
7,000b76558e,c3d3a8e8c6,d283080a93,f976a3c5f9,38c655d9b4,48.0,ea9e0181eb,8f55c680fa,0.450,0,0
8,00117bd32b,c3d3a8e8c6,ad2b2e17d2,eda7b2976b,1e7082b4bb,177.0,c34d31ceec,7e01a40bc1,0.250,0,0
9,0011ab9d97,e344ab2e71,ed2ad1797c,b25baa9dd5,fe255b600a,116.0,fce7934ead,95ea122ece,0.005,0,0


In [19]:
import duckdb

con = duckdb.connect()

purchases_path = OUTPUT_DATA_DIR / "purchases.parquet"

purchases = con.read_parquet(str(purchases_path))

# Preview
display(purchases.limit(10).df())

# Schema
display(
    pd.DataFrame(
        {"column": purchases.columns, "dtype": [str(t) for t in purchases.types]}
    )
)

,client_id,transaction_id,transaction_datetime,regular_points_received,express_points_received,regular_points_spent,express_points_spent,purchase_sum,store_id,product_id,product_quantity,trn_sum_from_iss,trn_sum_from_red
0,000012768d,7e3e2e3984,2018-12-01 07:12:45,10.0,0.0,0.0,0.0,1007.0,54a4a11a29,9a80204f78,2.0,80.0,NaN
1,000012768d,7e3e2e3984,2018-12-01 07:12:45,10.0,0.0,0.0,0.0,1007.0,54a4a11a29,da89ebd374,1.0,65.0,NaN
2,000012768d,7e3e2e3984,2018-12-01 07:12:45,10.0,0.0,0.0,0.0,1007.0,54a4a11a29,0a95e1151d,1.0,24.0,NaN
3,000012768d,7e3e2e3984,2018-12-01 07:12:45,10.0,0.0,0.0,0.0,1007.0,54a4a11a29,4055b15e4a,2.0,50.0,NaN
4,000012768d,7e3e2e3984,2018-12-01 07:12:45,10.0,0.0,0.0,0.0,1007.0,54a4a11a29,a685f1916b,1.0,22.0,NaN
5,000012768d,7e3e2e3984,2018-12-01 07:12:45,10.0,0.0,0.0,0.0,1007.0,54a4a11a29,21db5dbe53,1.0,34.0,NaN
6,000012768d,7e3e2e3984,2018-12-01 07:12:45,10.0,0.0,0.0,0.0,1007.0,54a4a11a29,1e208d0b4c,1.0,24.0,NaN
7,000012768d,7e3e2e3984,2018-12-01 07:12:45,10.0,0.0,0.0,0.0,1007.0,54a4a11a29,15ccaa8685,1.0,51.0,NaN
8,000012768d,7e3e2e3984,2018-12-01 07:12:45,10.0,0.0,0.0,0.0,1007.0,54a4a11a29,45389bb5b0,1.0,23.0,NaN
9,000012768d,7e3e2e3984,2018-12-01 07:12:45,10.0,0.0,0.0,0.0,1007.0,54a4a11a29,cb4c804130,1.0,60.0,NaN


,column,dtype
0,client_id,VARCHAR
1,transaction_id,VARCHAR
2,transaction_datetime,TIMESTAMP
3,regular_points_received,DOUBLE
4,express_points_received,DOUBLE
5,regular_points_spent,DOUBLE
6,express_points_spent,DOUBLE
7,purchase_sum,DOUBLE
8,store_id,VARCHAR
9,product_id,VARCHAR
